<a href="https://colab.research.google.com/github/Ganasa18/belajar-tensorflow/blob/main/train_class_prompt_labeling_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# 1 — GOOGLE DRIVE PERSISTENT STORAGE SETUP
# ROUTER AUTO TUNE V2

from google.colab import drive

import os
import shutil


# =========================================================
# 1. MOUNT GOOGLE DRIVE
# =========================================================

DRIVE_MOUNT = "/content/drive"

drive_ok = False

try:

    drive.mount(
        DRIVE_MOUNT,
        force_remount=False
    )

    assert os.path.exists(
        f"{DRIVE_MOUNT}/MyDrive"
    ), "Google Drive belum mounted"

    drive_ok = True

except Exception as e:

    print()
    print("WARNING: Google Drive gagal dimount.")
    print(type(e).__name__, str(e))

    drive_ok = False


# =========================================================
# 2. STORAGE DIRECTORY
# =========================================================

if drive_ok:

    SAVE_DIR = (
        f"{DRIVE_MOUNT}/MyDrive/router_classifier"
    )

else:

    print()
    print(
        "Fallback sementara ke /content."
    )

    print(
        "WARNING: data bisa hilang "
        "jika runtime restart."
    )

    SAVE_DIR = (
        "/content/router_classifier_temp"
    )


os.makedirs(
    SAVE_DIR,
    exist_ok=True
)


# =========================================================
# 3. V1 BASELINE PATHS
#
# File lama hanya dibaca sebagai baseline.
# Jangan overwrite dari flow V2.
# =========================================================

BASE_PATH = (
    f"{SAVE_DIR}/router_teacher_checkpoint.csv"
)

EXTRA_ALL_PATH = (
    f"{SAVE_DIR}/router_teacher_extra_all.csv"
)

CURRENT_DATASET_PATH = (
    f"{SAVE_DIR}/router_dataset_current.csv"
)

V1_MODEL_PATH = (
    f"{SAVE_DIR}/router_classifier_current.joblib"
)

V1_ROUND_LOG_PATH = (
    f"{SAVE_DIR}/router_tuning_rounds.csv"
)


# =========================================================
# 4. ROUTER V2 PATHS
# =========================================================

V2_ACCEPTED_PATH = (
    f"{SAVE_DIR}/router_v2_accepted.csv"
)

V2_REJECTED_PATH = (
    f"{SAVE_DIR}/router_v2_rejected.csv"
)

V2_HISTORY_PATH = (
    f"{SAVE_DIR}/router_v2_history.csv"
)

V2_VALIDATION_PATH = (
    f"{SAVE_DIR}/router_v2_validation_fixed.csv"
)

V2_CURRENT_MODEL_PATH = (
    f"{SAVE_DIR}/router_v2_current.joblib"
)

V2_BEST_MODEL_PATH = (
    f"{SAVE_DIR}/router_v2_best.joblib"
)

V2_BEST_METRICS_PATH = (
    f"{SAVE_DIR}/router_v2_best_metrics.json"
)

V2_ACTIVE_DATASET_PATH = (
    f"{SAVE_DIR}/router_v2_active_dataset.csv"
)


# =========================================================
# 5. STORAGE STATUS
# =========================================================

print()
print("=" * 65)
print("ROUTER V2 STORAGE STATUS")
print("=" * 65)

print(
    "Drive mounted :",
    drive_ok
)

print(
    "SAVE_DIR      :",
    SAVE_DIR
)

print()

print(
    "V1 baseline dataset :",
    CURRENT_DATASET_PATH
)

print(
    "V2 accepted data    :",
    V2_ACCEPTED_PATH
)

print(
    "V2 rejected data    :",
    V2_REJECTED_PATH
)

print(
    "V2 validation       :",
    V2_VALIDATION_PATH
)

print(
    "V2 best model       :",
    V2_BEST_MODEL_PATH
)

print("=" * 65)

Mounted at /content/drive

ROUTER V2 STORAGE STATUS
Drive mounted : True
SAVE_DIR      : /content/drive/MyDrive/router_classifier

V1 baseline dataset : /content/drive/MyDrive/router_classifier/router_dataset_current.csv
V2 accepted data    : /content/drive/MyDrive/router_classifier/router_v2_accepted.csv
V2 rejected data    : /content/drive/MyDrive/router_classifier/router_v2_rejected.csv
V2 validation       : /content/drive/MyDrive/router_classifier/router_v2_validation_fixed.csv
V2 best model       : /content/drive/MyDrive/router_classifier/router_v2_best.joblib


In [2]:
# =========================================================
# CELL 2 — DEPENDENCIES + IMPORTS
# ROUTER AUTO TUNE V2
# =========================================================

import os
import re
import json
import time
import random
import joblib

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV,
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)


# =========================================================
# RANDOM SEED
# =========================================================

RANDOM_STATE = 42

random.seed(
    RANDOM_STATE
)

np.random.seed(
    RANDOM_STATE
)


# =========================================================
# GLOBAL CONFIG
# =========================================================

MIN_CONFIDENCE = 0.80

MIN_IMPROVEMENT = 0.005

MAX_CLASS_RECALL_DROP = 0.05

CANDIDATE_BATCH_SIZE = 6

MAX_NO_IMPROVE_ROUNDS = 4


GRID_C_VALUES = [
    0.05,
    0.10,
    0.30,
    0.50,
    1.00,
    2.00,
    3.00,
    5.00,
    10.00,
]


# =========================================================
# CHECK
# =========================================================

print()
print("=" * 65)
print("ROUTER V2 ENVIRONMENT")
print("=" * 65)

print(
    "RANDOM_STATE          :",
    RANDOM_STATE
)

print(
    "MIN_CONFIDENCE        :",
    MIN_CONFIDENCE
)

print(
    "MIN_IMPROVEMENT       :",
    MIN_IMPROVEMENT
)

print(
    "CANDIDATE_BATCH_SIZE  :",
    CANDIDATE_BATCH_SIZE
)

print(
    "MAX_NO_IMPROVE_ROUNDS:",
    MAX_NO_IMPROVE_ROUNDS
)

print("=" * 65)


ROUTER V2 ENVIRONMENT
RANDOM_STATE          : 42
MIN_CONFIDENCE        : 0.8
MIN_IMPROVEMENT       : 0.005
CANDIDATE_BATCH_SIZE  : 6
MAX_NO_IMPROVE_ROUNDS: 4


In [3]:
# =========================================================
# 3 — ROUTING LABEL DEFINITIONS
# ROUTER AUTO TUNE V2
# =========================================================

LABELS = [
    "SIMPLE",
    "GENERAL",
    "REASONING",
    "CODING_SIMPLE",
    "CODING_COMPLEX",
    "TRANSFORM",
    "CREATIVE",
]


CATEGORY_DESCRIPTIONS = {

    "SIMPLE":
        "Simple factual questions, definitions, easy lookup, "
        "basic calculations, or very short tasks requiring "
        "little to no reasoning.",

    "GENERAL":
        "Normal assistant questions and explanations that "
        "require some understanding but do not require "
        "substantial multi-step reasoning, comparison, "
        "planning, or software engineering analysis.",

    "REASONING":
        "Non-coding analytical tasks requiring comparison, "
        "planning, trade-offs, deduction, mathematics, "
        "decision making, or multi-step reasoning. "
        "Technology comparisons without actual software "
        "debugging or implementation belong here.",

    "CODING_SIMPLE":
        "Small programming tasks such as regex, SQL, syntax "
        "correction, short scripts, small functions, simple "
        "code modification, or straightforward implementation.",

    "CODING_COMPLEX":
        "Software engineering tasks requiring substantial "
        "technical reasoning involving debugging, architecture, "
        "security, concurrency, performance, databases, "
        "distributed systems, backend/frontend systems, "
        "or multi-component implementation.",

    "TRANSFORM":
        "Tasks whose main purpose is transforming existing "
        "user-provided content, including translation, "
        "summarization, rewriting, extraction, formatting, "
        "shortening, restructuring, or conversion.",

    "CREATIVE":
        "Creative generation and ideation such as stories, "
        "dialogue, slogans, names, concepts, fictional content, "
        "brainstorming, characters, scenarios, or creative variations.",
}


# =========================================================
# BOUNDARY MAP
# =========================================================

CONFUSION_BOUNDARIES = {

    "SIMPLE": [
        "GENERAL",
        "TRANSFORM",
        "CREATIVE",
        "REASONING",
    ],

    "GENERAL": [
        "SIMPLE",
        "REASONING",
        "CODING_COMPLEX",
    ],

    "REASONING": [
        "CODING_COMPLEX",
        "GENERAL",
    ],

    "CODING_SIMPLE": [
        "CODING_COMPLEX",
        "REASONING",
    ],

    "CODING_COMPLEX": [
        "REASONING",
        "GENERAL",
        "CODING_SIMPLE",
    ],

    "TRANSFORM": [
        "SIMPLE",
        "CREATIVE",
    ],

    "CREATIVE": [
        "TRANSFORM",
        "SIMPLE",
    ],
}


# =========================================================
# VALIDATION
# =========================================================

for label in LABELS:

    if label not in CATEGORY_DESCRIPTIONS:

        raise ValueError(
            f"Missing category description: {label}"
        )


print()
print("=" * 65)
print("ROUTING TAXONOMY")
print("=" * 65)

for i, label in enumerate(
    LABELS,
    start=1
):

    print(
        f"{i}. {label}"
    )

print()
print(
    "Total labels:",
    len(LABELS)
)

print("=" * 65)


ROUTING TAXONOMY
1. SIMPLE
2. GENERAL
3. REASONING
4. CODING_SIMPLE
5. CODING_COMPLEX
6. TRANSFORM
7. CREATIVE

Total labels: 7


In [4]:
# =========================================================
# CELL 4 — LOAD EMBEDDING MODEL
# ROUTER AUTO TUNE V2
# =========================================================

from sentence_transformers import SentenceTransformer


# =========================================================
# MODEL CONFIG
# =========================================================

EMBEDDING_MODEL_NAME = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)


# =========================================================
# LOAD MODEL
# =========================================================

print()
print("=" * 65)
print("LOADING EMBEDDING MODEL")
print("=" * 65)

print(
    "Model:",
    EMBEDDING_MODEL_NAME
)


embedding_model = (
    SentenceTransformer(
        EMBEDDING_MODEL_NAME
    )
)


print()
print(
    "Embedding model loaded ✅"
)

print("=" * 65)


# =========================================================
# QUICK TEST
# =========================================================

test_embedding = (
    embedding_model.encode(
        [
            "apa itu HTTP",
            "buat function python sederhana",
        ],
        normalize_embeddings=True
    )
)


print(
    "Embedding shape:",
    test_embedding.shape
)

print(
    "Embedding dimension:",
    test_embedding.shape[1]
)

print(
    "Finite values:",
    bool(
        np.isfinite(
            test_embedding
        ).all()
    )
)


LOADING EMBEDDING MODEL
Model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Embedding model loaded ✅
Embedding shape: (2, 384)
Embedding dimension: 384
Finite values: True


In [ ]:
# Cell 5 — Teacher API + Parser

import requests
import time
import re
import json

# Konfigurasi API teacher
import os

BASE_URL = "https://openrouter.ai/api/v1/chat/completions"
API_KEY = "ISI_API_KEY"
MODEL = "openrouter/free"

TEMPERATURE = 0
TIMEOUT = 90

from google.colab import userdata
API_KEY = userdata.get('OPEN_ROUTER')



# Call Api

def call_teacher(prompt):
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": MODEL,
        "temperature": 0,
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    }

    print("  -> POST", BASE_URL)
    print("  -> model:", MODEL)

    start = time.time()

    response = requests.post(
        BASE_URL,
        headers=headers,
        json=payload,

        # jangan 90 detik dulu untuk debugging
        timeout=(10, 30)
    )

    print(
        f"  <- HTTP {response.status_code}"
        f" ({time.time() - start:.2f}s)"
    )

    response.raise_for_status()

    data = response.json()

    return data["choices"][0]["message"]["content"]


# Parser JSON yang lebih tahan error



def parse_teacher_output(text):
    if not text:
        raise ValueError("Teacher mengembalikan response kosong")

    text = text.strip()

    # Hapus markdown fence kalau model memberi ```json
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)

    # Cari object JSON
    match = re.search(r'\{[\s\S]*?\}', text)

    if not match:
        raise ValueError(
            f"Teacher tidak mengembalikan JSON. Raw: {text[:300]!r}"
        )

    try:
        data = json.loads(match.group())
    except json.JSONDecodeError as e:
        raise ValueError(
            f"JSON teacher rusak: {match.group()[:300]}"
        ) from e

    label = data.get("label")

    try:
        difficulty = int(data.get("difficulty"))
        confidence = float(data.get("confidence"))
    except (TypeError, ValueError):
        raise ValueError(
            f"difficulty/confidence invalid: {data}"
        )

    if label not in LABELS:
        raise ValueError(f"Label invalid: {label}")

    if not 1 <= difficulty <= 5:
        raise ValueError(f"Difficulty invalid: {difficulty}")

    if not 0 <= confidence <= 1:
        raise ValueError(f"Confidence invalid: {confidence}")

    return {
        "label": label,
        "difficulty": difficulty,
        "confidence": confidence,
    }